# Lab 10: Report

## Tensile Testing of Polymers

The main objectives of this lab are:

- Test the mechanical properties of polymers in tension with the Instron tensile test machine.
- Observe the influence of deformation rate and temperature on the mechanical properties.

Recommended reading: Callister, *Materials Science and Engineering: An Introduction*, Chapter 14 to Section 15.9.

Run the cell below to import the necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Polymer structures (10 points)

**`YOUR RESPONSE`**: Draw a 2D representation of all polymer monomer units (showing elements and bonds) and the schematic arrangements of chains in:
- low density poly(ethylene): LDPE
- high density poly(ethylene): HDPE
- poly(carbonate): PC
- poly(methyl methacrylate): PMMA

You can make your drawing by hand or with software. Save your drawing as an image file in the same folder as this notebook and display it with the code cell below.

In [ ]:
# Display your drawing of the polymer structures
structure_image = plt.imread("image_file.ext")      # STUDENT: replace the "image_file.ext" string with your file

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(structure_image)
ax.axis("off")
plt.show()

# STUDENT: provide a descriptive caption
print("Figure 1. YOUR RESPONSE")

## 2. Import the data

Each polymer has its own `.csv` file exported by the Instron tensile tester.

1. Open one of the `.csv` files (in Excel or a text editor) and inspect it.
    - **Results Table 1** lists every specimen tested, with its dimensions and the crosshead (deformation) rate.
    - **Results Table 2** contains one block of measurements per specimen: time, displacement, and force.
1. Note the **units** of every quantity. You will need them in Section 3.
1. Place the four `.csv` files in the same folder as this notebook and update `FILES` below to match the file names.

In [ ]:
# STUDENT: update the file names to match your data files
FILES = {
    "HDPE": "filename.csv",
    "LDPE": "filename.csv",
    "PC":   "filename.csv",
    "PMMA": "filename.csv",
}


### 2.1 Read the files

The code cell below defines a function that reads one Instron `.csv` file, then uses it to read all four files. Run the cell without changing it.

For each specimen in a file, the function combines specimen data from **Results Table 1** with tensile test data from **Results Table 2**. 

| columns of `df` | contents |
|:---|:---|
| `Material`, `Specimen` | specimen information |
| `Temperature / °C` | test temperature |
| `Thickness / mm`, `Width / mm`, `Length / mm` | specimen dimensions |
| `Rate / mm/min` | crosshead rate |
| `Time / s`, `Displacement / mm`, `Force / kN` | tensile test measurements |

The specimen information is the same on every row of a specimen, so each measurement row carries everything needed to calculate its stress and strain.

In [ ]:
def read_instron_file(path, material):
    # The rows of the file have different lengths, so read every cell as text into 6 columns
    raw = pd.read_csv(path, header=None, names=range(6), dtype=str)

    # Results Table 1: the specimen rows sit between the units row and "Results Table 2"
    table2_row = raw.index[raw[0] == "Results Table 2"][0]
    specimens = raw.iloc[3:table2_row].dropna(subset=[1])

    # Results Table 2: each block of measurements starts on a row whose column 1 says "Time"
    block_starts = list(raw.index[raw[1] == "Time"]) + [len(raw)]

    curves = []
    for k in range(len(specimens)):
        specimen = specimens.iloc[k]                                                   # row k of Results Table 1
        block = raw.iloc[block_starts[k] + 2 : block_starts[k + 1], 1:4].dropna()     # skip the header and units rows

        curves.append(pd.DataFrame({
            # specimen information: one value, repeated on every row of this specimen
            "Material":          material,
            "Specimen":          specimen[1],
            "Temperature / °C":  50 if "50C" in specimen[1] else 21,
            "Thickness / mm":    float(specimen[2]),
            "Width / mm":        float(specimen[3]),
            "Length / mm":       float(specimen[4]),
            "Rate / mm/min":     float(specimen[5]),
            # measurements: one value per row
            "Time / s":          pd.to_numeric(block[1]).to_numpy(),
            "Displacement / mm": pd.to_numeric(block[2]).to_numpy(),
            "Force / kN":        pd.to_numeric(block[3]).to_numpy(),
        }))

    return pd.concat(curves, ignore_index=True)


The code below runs the function `read_instron_file` on each `.csv` file.

The tables from the four files are then stacked into a single table, `df`, with one row per measurement. 

In [ ]:
df = []
for material, path in FILES.items():
    df.append(read_instron_file(path, material))

df = pd.concat(df, ignore_index=True)

print("Compiled Data Frame called df")
display(df)

print("Specimen Summary:")
specimen_summary = df.drop_duplicates(subset="Specimen").reset_index(drop=True)
display(specimen_summary[["Specimen", "Thickness / mm", "Width / mm", "Length / mm", "Rate / mm/min"]])

## 3. Calculations

**`YOUR RESPONSE`**: In this lab you will use **engineering** stress and strain. 

Write the equations for engineering stress and engineering strain in LaTeX and define every symbol.

$$
\sigma = \ldots
$$

$$
\epsilon = \ldots
$$

### 3.1 Calculate stress and strain

In the code below:

1. convert crosshead rate to in/min
1. calculate engineering strain
1. calculate engineering stress in MPa
1. ensure all values are rounded to appropriate significant figures

**Variables for your code**:

- `df["Rate / mm/min"]`: crosshead rate
- `df["Displacement / mm"]`: crosshead displacement
- `df["Force / kN"]`: crosshead force
- `df["Thickness / mm"]`: specimen thickness
- `df["Width / mm"]`: specimen width
- `df["Length / mm"]`: specimen length

In [ ]:
# STUDENT: crosshead rate in in/min, rounded to 2 decimal places
df["Rate / in/min"] = YOUR RESPONSE

# STUDENT: engineering strain
df["Strain"] = YOUR RESPONSE

# STUDENT: engineering stress in MPa
df["Stress / MPa"] = YOUR RESPONSE

display(df[["Specimen", "Rate / in/min", "Strain", "Stress / MPa"]])

## 4. Plotting (20 points)

### 4.1 Stress–strain curves of HDPE, LDPE, and PC 

**`YOUR RESPONSE`**: Plot the engineering stress–engineering strain curves for HDPE, LDPE, and PC and find the maximum (yield) strength and the flow stress.

First, set your plotting choices. `COLOR` sets one color per material. `LINESTYLES` sets the line style of each specimen of a material, from the slowest to the fastest deformation rate.

In [ ]:
# STUDENT: choose a color for each material and a line style for each deformation rate. They are all the same by default
COLOR = {"HDPE": "k", "LDPE": "k", "PC": "k", "PMMA": "k"}
LINESTYLES = ["-", "-", "-", "-"]      # slowest → fastest rate

Complete the plotting code below. The figure has one subplot per material.

**Variables available inside the inner loop** (for one specimen at a time):

- `curve["Strain"]`, `curve["Stress / MPa"]`: x and y values
- `rate`: deformation rate of the specimen in in/min
- `linestyle`: the line style for this specimen

**Functions**:

- `ax.plot(x, y, color=..., linestyle=..., label=...)`: plot a line
- `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()`, `ax.grid()`, `ax.legend()`

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

if "df_plot" not in globals(): df_plot = df.copy()

for ax, material in zip(axes, ["HDPE", "LDPE", "PC"]):
    curves = df_plot[df_plot["Material"] == material].groupby("Specimen", sort=False)

    for (specimen, curve), linestyle in zip(curves, LINESTYLES):
        rate = curve["Rate / in/min"].iloc[0]

        # STUDENT: plot engineering stress vs engineering strain. Label each curve with its deformation rate
        YOUR RESPONSE

    # STUDENT: add a title, axis labels with units, grid, and legend to each subplot
    YOUR RESPONSE

plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption
print("Figure 2. YOUR RESPONSE")

Next, run the cell below to drop the fracture tail. You can adjust the parameters to change how much of the tail is cut off. After removing the tail data, re-run the plotting cell above.

In [ ]:
WINDOW = 3      # number of points over which the slope is measured
GAP = 0.02      # also remove GAP portion of strain before the steepest drop (the fracture descent)

drop = df.groupby("Specimen")["Stress / MPa"].diff(WINDOW)                              # change in stress from WINDOW points behind
steepest = drop.groupby(df["Specimen"]).transform("idxmin")                             # row of the steepest drop
fracture_strain = df.loc[steepest, "Strain"].to_numpy()                                 # strain at that row, on every row
fractured = drop.groupby(df["Specimen"]).transform("min") < -0.01 * WINDOW * df.groupby("Specimen")["Stress / MPa"].transform("max")
before = (df.index < steepest) & (df["Strain"] < (1 - GAP) * fracture_strain)          # before the fracture descent
df_plot = df[before | ~fractured]                                                       # cut only specimens that fractured

**`ENGINEERING CHECK`**

Can every curve be distinguished? Where does each curve reach its maximum (yield) stress, and where does it settle into a plateau of roughly constant flow stress? Does every specimen reach a flow plateau before it breaks?

When you're satisfied with the plot above, run the code cell below to delete the tail data from further analysis.

In [ ]:
# overwrite the df with the plotting data once satisfied with the tail removal
df = df_plot

### 4.2 Maximum (yield) strength

For a polymer that necks, the maximum of the engineering stress–strain curve is often taken as the yield strength.

Complete the code below to find the maximum stress of each specimen.

**Daisy chain together the following methods**:

- `groupby("COLUMN_ONE")["COLUMN_TWO"]`: group the data by COLUMN_ONE, keep only the data in COLUMN_TWO
- `.max()`: returns the maximum value (in this instance the maximum of each group)
- `.round(n)`: round the data to n decimal places

The result is stored in a new table, `results`, that will hold all the mechanical properties.

In [ ]:
# First create the "results" dataframe with one row per specimen, labeled by specimen name
results = df[["Material", "Specimen", "Temperature / °C", "Rate / in/min"]].drop_duplicates().set_index("Specimen")

# STUDENT: determine the maximum (yield) strength of each specimen, rounded to 1 decimal place
results["Yield strength / MPa"] = YOUR RESPONSE

display(results)

### 4.3 Flow stress

The flow stress is the roughly constant stress needed to keep the neck propagating along the specimen. Determine it as the **average stress over a strain window** in the plateau region of each curve.

1. Run the first cell below. It prints a template of `FLOW_WINDOWS` with the specimen names in your data. Copy it into the cell after it.
1. Complete the line that averages the stress inside the window, and run the cell. With every window still `(np.nan, np.nan)`, no flow stress is calculated yet.
1. Run the plotting cell. It shows every curve, including PMMA, in its own panel.
1. For each specimen, choose a strain window in the plateau and enter it in `FLOW_WINDOWS` as `(lower strain, upper strain)`. Leave `(np.nan, np.nan)` for specimens that do not reach a flow plateau.
1. Re-run the `FLOW_WINDOWS` cell and the plot, and check that each shaded window and flow-stress line sit on the plateau. Repeat as needed.

In [ ]:
# Run this cell, then copy the printed dictionary into the next cell
print("FLOW_WINDOWS = {")
for specimen in results.index:
    print(f'    "{specimen}": (np.nan, np.nan),')
print("}")

print("DELETE THIS CELL AFTER COPYING THE OUTPUT ABOVE")

# DELETE THIS CELL AFTER COPYING THE OUTPUT

Use the cell below to set the flow stress windows for the applicable materials.

Run the cell below and the subsequent plotting cell as many times as you like.

In [ ]:
# Enter (lower strain, upper strain) for every specimen that reaches a flow plateau
# Leave (np.nan, np.nan) for specimens without a plateau.
# STUDENT: Paste output from the cell above here, replacing this comment line


# give every row of df its specimen's (lower, upper) strain window, then mark the rows inside it
window = df["Specimen"].map(FLOW_WINDOWS)
in_window = df["Strain"].between(window.str[0], window.str[1])

# STUDENT: flow stress = average stress of the rows inside the window, for each specimen
results["Flow stress / MPa"] = YOUR RESPONSE

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

#skip panels with no data
for i in [3, 7, 11]: axes.ravel()[i].set_visible(False)   # hide the three unused panels
axes = np.delete(axes.ravel(), [3, 7, 11])                 # remove them from the list used in the loop

for ax, (specimen, info) in zip(axes, results.iterrows()):
    curve = df[df["Specimen"] == specimen]
    ax.plot(curve["Strain"], curve["Stress / MPa"], color=COLOR[info["Material"]])
    if not np.isnan(info["Flow stress / MPa"]):
        ax.axvspan(*FLOW_WINDOWS[specimen], color="gray", alpha=0.2)
        ax.axhline(info["Flow stress / MPa"], color="gray", linestyle="--")
    ax.set_title(f"{specimen} ({info['Rate / in/min']:g} in/min)", fontsize=10)
    ax.grid(alpha=0.3)

fig.supxlabel("Engineering strain", fontsize=14)
fig.supylabel("Engineering stress / MPa", fontsize=14)
plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption
print("Figure 3. YOUR RESPONSE")


### 4.4 Output results

In the cell below, use `display()` to show the `"Yield strength / MPa"` and `"Flow stress / MPa"` columns of the `results` dataframe.

In [ ]:
display(results[[YOUR RESPONSE]])

## 5. Stress–strain curves of PMMA (20 points)

**`YOUR RESPONSE`**: Plot the engineering stress–engineering strain curves for PMMA at different temperatures. How and why do they differ for different temperatures?

This time, write the plotting code yourself. Plot all four PMMA specimens on one set of axes, and make the specimens tested at room temperature and at 50 °C easy to tell apart.

**Hints**:

- `df[df["Material"] == "PMMA"].groupby("Specimen", sort=False)` loops over the PMMA specimens, as in Section 4.
- `curve["Temperature / °C"].iloc[0]` and `curve["Rate / in/min"].iloc[0]` give the temperature and rate of a specimen, for the legend.
- Include a title, axis labels with units, a grid, a legend, and a caption.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# STUDENT: plot the engineering stress-strain curves of the four PMMA specimens
for specimen, curve in df[df["Material"] == "PMMA"].groupby("Specimen", sort=False):
    temperature = curve["Temperature / °C"].iloc[0]      # the same on every row of the specimen, so take the first
    rate = curve["Rate / in/min"].iloc[0]
    # STUDENT: Add plotting code here

#STUDENT: Add title, axis labels, grid, legend

plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption
print("Figure 4. YOUR RESPONSE")

**`YOUR RESPONSE`**: How and why do the PMMA curves differ for different temperatures? Think about the structural changes with temperature in amorphous, non-crystalline materials.

*Replace this text with your response in paragraph form.*

## 6. Effect of deformation rate (30 points)

Complete the cell below to plot the yield stress vs. the deformation rate and the flow stresses vs. the deformation rate for all materials (if applicable) and temperatures.

Use one subplot for the yield strength and one for the flow stress. Plot each combination of material and temperature (HDPE, LDPE, PC, PMMA at room temperature, and PMMA at 50 °C) as its own set of connected markers. To compact the data, use a **logarithmic scale** for the deformation rate.

**Variables for your code** (for one material and temperature at a time):

- `group["Rate / in/min"]`: x values
- `group["Yield strength / MPa"]`, `group["Flow stress / MPa"]`: y values for each subplot
- `label`: the material and temperature, for the legend
- `ax_yield`: plotting handle for the yield strengths
- `ax_flow`: plotting handle for the flow stresses

**Functions**:

- `ax.plot(x, y, marker=..., label=...)`: plot connected markers
- `ax.set_xscale("log")`: logarithmic x-axis

In [ ]:
fig, (ax_yield, ax_flow) = plt.subplots(1, 2, figsize=(14, 5))

for (material, temperature), group in results.groupby(["Material", "Temperature / °C"], sort=False):
    group = group.sort_values("Rate / in/min")
    label = f"{material}, {temperature} °C"

    # STUDENT: generate plots of the yield strength (ax_yield) and the flow stress (ax_flow) vs the deformation rate
    YOUR RESPONSE

# STUDENT: use a logarithmic rate axis, and add titles, axis labels with units, grids, and legends to both subplots
YOUR RESPONSE

plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption
print("Figure 5. YOUR RESPONSE")

**`YOUR RESPONSE`**: Explain the differences in yield strength (at similar strain rates) of HDPE, LDPE, and PC, based on the microstructural arrangements (chain length, primary and secondary bonds, ...) and chemistry of the polymer chains.

*Replace this text with your response in paragraph form.*

**`YOUR RESPONSE`**: Explain the changes in flow stress with varying strain rate. Focus on the microstructural arrangements and primary and secondary bonds of the various polymers.

*Replace this text with your response in paragraph form.*

## CONTRIBUTION STATEMENT

*List each contributor and what they did.*

**`YOUR RESPONSE`**

| Contributor | Contribution |
|:---|:---|
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |

## SUBMISSION

When this notebook is complete and ready to submit, export it as a PDF file. If the PDF export fails, check your $\LaTeX$ code for errors.